# 07_predict_v3

Generates daily predictions for all stocks in the Precursor pipeline.
Runs after 04_gold.py completes.

**Changes from v2:**
- Dual horizon setup:
    XGBoost → target_21d (monthly direction, AUC 0.5333)
    TFT     → target_1d  (daily direction, trained on target_1d)
- load_features() loads BOTH target_1d and target_21d
- Separate set_a/set_b per model based on their target column
- horizon column added to predictions table ("1d" or "21d")
- Backtest uses target_21d for XGBoost, target_1d for TFT
- Agreement signal computed: both UP / both DOWN / disagree

In [0]:
# CELL 1 — Install libraries

%pip install xgboost scikit-learn "numpy<2.0" pytorch-lightning pytorch-forecasting -q 
%pip install torch --index-url https://download.pytorch.org/whl/cpu -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# CELL 2 — Imports and setup

import pickle
import json
import pandas as pd
import numpy as np
import logging
from datetime import datetime
from typing import Optional

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DateType, DoubleType, IntegerType,
    TimestampType, BooleanType,
)

import xgboost as xgb
import torch
import pytorch_lightning
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data.encoders import NaNLabelEncoder

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.predict")

MODEL_VOLUME   = "/Volumes/precursor/models/artifacts"
BACKTEST_START = "2024-01-01"

# Dual horizon targets
XGB_TARGET = "target_21d"   # XGBoost: monthly direction
TFT_TARGET = "target_1d"    # TFT:     daily direction

FEATURE_COLS = [
    "return_1d", "return_5d", "return_21d", "return_63d",
    "volatility_21d", "volatility_63d",
    "volume_zscore_21d",
    "price_vs_52w_high", "price_vs_52w_low",
    "price_vs_sma20", "price_vs_sma50", "price_vs_sma200",
    "rsi_14", "macd", "macd_signal", "macd_histogram",
    "bb_width", "bb_position",
    "obv_5d_sum", "obv_21d_sum",
    "atr_pct",
    "fed_rate_delta_21d", "fed_rate_delta_63d",
    "yield_curve_level", "yield_curve_change_21d",
    "vix_level", "vix_zscore_63d",
    "inflation_mom", "unemployment_delta_63d",
    "m2_growth_63d", "macro_regime",
    "insider_filings_7d", "insider_filings_30d",
    "insider_filings_90d",
    "insider_activity_spike",
    "days_since_last_filing",
]

TIME_VARYING_KNOWN = [
    "fed_rate_delta_21d", "fed_rate_delta_63d",
    "yield_curve_level", "yield_curve_change_21d",
    "vix_level", "vix_zscore_63d",
    "inflation_mom", "unemployment_delta_63d",
    "m2_growth_63d", "macro_regime",
]
TIME_VARYING_UNKNOWN = [
    "return_1d", "return_5d", "return_21d", "return_63d",
    "volatility_21d", "volume_zscore_21d",
    "rsi_14", "macd", "bb_position", "atr_pct",
    "insider_filings_7d", "insider_filings_30d",
    "insider_activity_spike",
]

logger.info(
    "precursor.predict v3 — XGBoost:%s TFT:%s numpy:%s",
    XGB_TARGET, TFT_TARGET, np.__version__
)

INFO:precursor.predict:precursor.predict v3 — XGBoost:target_21d TFT:target_1d numpy:1.23.5


In [0]:
# CELL 3 — Load data
# Loads both target columns and creates separate sets per model

def load_features() -> tuple[
    pd.DataFrame, pd.DataFrame,
    pd.DataFrame, pd.DataFrame,
]:
    """Load gold features with both target columns.

    Returns four DataFrames:
        set_a_xgb: historical rows for XGBoost (target_21d not null)
        set_b_xgb: inference rows for XGBoost  (target_21d is null)
        set_a_tft: historical rows for TFT      (target_1d not null)
        set_b_tft: inference rows for TFT       (target_1d is null)
    """
    cols = (
        ["ticker", "date", "sector"] +
        FEATURE_COLS +
        ["target_1d", "target_21d"]
    )

    base = spark.table("precursor.gold.features").select(*cols)

    def to_pdf(sdf: "pyspark.sql.DataFrame") -> pd.DataFrame:
        pdf = sdf.toPandas()
        pdf["date"] = pd.to_datetime(pdf["date"])
        pdf[FEATURE_COLS] = pdf[FEATURE_COLS].fillna(0.0)
        return pdf

    # XGBoost sets — based on target_21d
    set_a_xgb = to_pdf(
        base.filter(
            f"date >= '{BACKTEST_START}' AND target_21d IS NOT NULL"
        ).orderBy("ticker", "date")
    )
    set_b_xgb = to_pdf(
        base.filter("target_21d IS NULL")
            .orderBy("ticker", "date")
    )

    # TFT sets — based on target_1d
    set_a_tft = to_pdf(
        base.filter(
            f"date >= '{BACKTEST_START}' AND target_1d IS NOT NULL"
        ).orderBy("ticker", "date")
    )
    set_b_tft = to_pdf(
        base.filter("target_1d IS NULL")
            .orderBy("ticker", "date")
    )

    logger.info("XGBoost set_a: %d rows  set_b: %d rows",
                len(set_a_xgb), len(set_b_xgb))
    logger.info("TFT     set_a: %d rows  set_b: %d rows",
                len(set_a_tft), len(set_b_tft))

    if len(set_b_xgb) > 0:
        logger.info("XGBoost inference dates: %s → %s",
                    set_b_xgb["date"].min().date(),
                    set_b_xgb["date"].max().date())
    if len(set_b_tft) > 0:
        logger.info("TFT inference dates: %s → %s",
                    set_b_tft["date"].min().date(),
                    set_b_tft["date"].max().date())

    return set_a_xgb, set_b_xgb, set_a_tft, set_b_tft

In [0]:
# CELL 4 — Load models

def load_xgboost() -> tuple[xgb.XGBClassifier, bool]:
    """Load XGBoost bundle. Returns (model, inverted_flag)."""
    path = f"{MODEL_VOLUME}/xgb_baseline.pkl"
    with open(path, "rb") as f:
        bundle = pickle.load(f)

    if isinstance(bundle, dict):
        model    = bundle["model"]
        inverted = bundle.get("inverted", False)
        auc      = bundle.get("auc", 0.0)
        target   = bundle.get("target", "unknown")
        logger.info(
            "XGBoost bundle loaded: target=%s auc=%.4f inverted=%s numpy=%s",
            target, auc, inverted, bundle.get("numpy_version", "?")
        )
    else:
        model    = bundle
        inverted = False
        logger.warning("XGBoost loaded as raw model (old format)")

    return model, inverted


def load_tft() -> Optional[TemporalFusionTransformer]:
    """Load TFT checkpoint. Returns None on failure — non-fatal."""
    path = f"{MODEL_VOLUME}/tft_checkpoint.ckpt"
    try:
        model = TemporalFusionTransformer.load_from_checkpoint(path)
        model.eval()
        logger.info("TFT loaded from %s", path)
        return model
    except Exception as exc:
        logger.warning("TFT failed to load: %s", exc)
        return None

In [0]:
# CELL 5 — XGBoost predictions (horizon = 21d)

def predict_xgboost(
    model:        xgb.XGBClassifier,
    inverted:     bool,
    df:           pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    """Generate XGBoost 21-day direction predictions.

    Args:
        model:        Trained XGBClassifier (trained on target_21d).
        inverted:     Whether to invert probabilities.
        df:           Feature DataFrame.
        dataset_name: "historical" or "inference".

    Returns:
        DataFrame with horizon="21d" predictions.
    """
    X     = df[FEATURE_COLS]
    probs = model.predict_proba(X)[:, 1]

    if inverted:
        probs = 1.0 - probs

    preds      = (probs >= 0.5).astype(int)
    confidence = np.abs(probs - 0.5) * 2

    result = pd.DataFrame({
        "ticker":       df["ticker"].values,
        "date":         df["date"].values,
        "model":        "xgboost",
        "horizon":      "21d",
        "prediction":   preds,
        "probability":  probs,
        "confidence":   confidence,
        "predicted_at": datetime.utcnow(),
        "dataset":      dataset_name,
    })

    logger.info("XGBoost %s [21d]: %d predictions", dataset_name, len(result))
    logger.info("  UP: %.1f%%  DOWN: %.1f%%  avg_conf: %.3f",
                (preds == 1).mean() * 100,
                (preds == 0).mean() * 100,
                confidence.mean())
    logger.info("  prob min: %.4f  max: %.4f  std: %.6f",
                probs.min(), probs.max(), probs.std())

    return result

In [0]:
# CELL 6 — TFT predictions (horizon = 1d)

def predict_tft(
    model:        Optional[TemporalFusionTransformer],
    df:           pd.DataFrame,
    dataset_name: str,
) -> Optional[pd.DataFrame]:
    """Generate TFT 1-day direction predictions.

    TFT was trained on target_1d so this is its native target.
    Uses threshold 0.3 (not 0.5) because TFT is conservative
    and clusters probabilities near 0.5.

    For inference rows, prepends 63 days of history per ticker
    so TFT has enough context to make predictions.

    Args:
        model:        Loaded TFT (trained on target_1d).
        df:           Feature DataFrame with target_1d column.
        dataset_name: "historical" or "inference".

    Returns:
        DataFrame with horizon="1d" predictions, or None on failure.
    """
    if model is None:
        return None

    try:
        # For inference rows TFT needs 63 days of history per ticker
        # because max_encoder_length=63 was used during training
        # Without this context TFT cannot make any predictions
        if dataset_name == "inference":
            feature_cols_sql = ", ".join(FEATURE_COLS)
            history = (
                spark.sql(f"""
                    SELECT ticker, date, sector, target_1d,
                           {feature_cols_sql}
                    FROM precursor.gold.features
                    WHERE target_1d IS NOT NULL
                    QUALIFY ROW_NUMBER() OVER (
                        PARTITION BY ticker
                        ORDER BY date DESC
                    ) <= 63
                    ORDER BY ticker, date
                """)
                .toPandas()
            )
            history["date"] = pd.to_datetime(history["date"])
            history[FEATURE_COLS] = history[FEATURE_COLS].fillna(0.0)
            df = pd.concat([history, df], ignore_index=True)
            df = df.drop_duplicates(subset=["ticker", "date"])
            df = df.sort_values(["ticker", "date"])

        df = df.copy()
        df = df.sort_values(["ticker", "date"])
        df["time_idx"] = df.groupby("ticker").cumcount()
        df["ticker"]   = df["ticker"].astype(str)
        df["sector"]   = df["sector"].astype(str) \
            if "sector" in df.columns else "Unknown"
        df[TFT_TARGET] = df[TFT_TARGET].fillna(0).astype(int)

        dataset = TimeSeriesDataSet.from_parameters(
            model.dataset_parameters,
            df,
            predict=True,
            stop_randomization=True,
        )

        dataloader = dataset.to_dataloader(
            train=False,
            batch_size=128,
            num_workers=0,
        )

        with torch.no_grad():
            raw = model.predict(dataloader, mode="raw", return_x=True)

        raw_preds = raw.output.prediction
        if hasattr(raw_preds, "numpy"):
            raw_preds = raw_preds.numpy()
        raw_preds = np.array(raw_preds).flatten()

        # Threshold 0.3 not 0.5 — TFT is conservative
        # probabilities cluster near 0.5 due to training dynamics
        # lower threshold captures more UP predictions
        probs      = raw_preds.clip(0, 1)
        preds      = (probs >= 0.3).astype(int)
        confidence = np.abs(probs - 0.5) * 2

        n = len(probs)
        result = pd.DataFrame({
            "ticker":       df["ticker"].values[-n:],
            "date":         df["date"].values[-n:],
            "model":        "tft",
            "horizon":      "1d",
            "prediction":   preds,
            "probability":  probs,
            "confidence":   confidence,
            "predicted_at": datetime.utcnow(),
            "dataset":      dataset_name,
        })

        logger.info("TFT %s [1d]: %d predictions", dataset_name, len(result))
        logger.info(
            "  UP: %.1f%%  DOWN: %.1f%%  avg_conf: %.3f",
            (preds == 1).mean() * 100,
            (preds == 0).mean() * 100,
            confidence.mean(),
        )

        return result

    except Exception as exc:
        logger.warning("TFT prediction failed: %s", exc)
        return None

In [0]:
# CELL 7 — Compute agreement signal

def compute_agreement(
    xgb_preds: pd.DataFrame,
    tft_preds: Optional[pd.DataFrame],
) -> Optional[pd.DataFrame]:
    """Compute agreement between XGBoost (21d) and TFT (1d) predictions.

    Agreement signal logic:
        both_up:   XGBoost=1 AND TFT=1 → strong bullish
        both_down: XGBoost=0 AND TFT=0 → strong bearish
        disagree:  models conflict      → uncertain

    This is a novel signal — when both models agree across
    different horizons, conviction is higher.

    Returns:
        DataFrame with columns: ticker, date, agreement, predicted_at
    """
    if tft_preds is None:
        logger.info("TFT not available — skipping agreement computation")
        return None

    try:
        xgb_inf = xgb_preds[xgb_preds["dataset"] == "inference"][
            ["ticker", "date", "prediction"]
        ].rename(columns={"prediction": "xgb_pred"})

        tft_inf = tft_preds[tft_preds["dataset"] == "inference"][
            ["ticker", "date", "prediction"]
        ].rename(columns={"prediction": "tft_pred"})

        merged = xgb_inf.merge(tft_inf, on=["ticker", "date"], how="inner")

        merged["agreement"] = np.where(
            (merged["xgb_pred"] == 1) & (merged["tft_pred"] == 1), "both_up",
            np.where(
                (merged["xgb_pred"] == 0) & (merged["tft_pred"] == 0), "both_down",
                "disagree"
            )
        )
        merged["predicted_at"] = datetime.utcnow()

        agreement_counts = merged["agreement"].value_counts().to_dict()
        logger.info("Agreement signal: %s", agreement_counts)

        return merged[["ticker", "date", "agreement", "predicted_at"]]

    except Exception as exc:
        logger.warning("Agreement computation failed: %s", exc)
        return None

In [0]:
# CELL 8 — Write predictions

def write_predictions(
    xgb_historical: pd.DataFrame,
    xgb_inference:  Optional[pd.DataFrame],
    tft_historical: Optional[pd.DataFrame],
    tft_inference:  Optional[pd.DataFrame],
    agreement_df:   Optional[pd.DataFrame],
) -> None:
    """Write all predictions to Delta tables.

    Writes:
        precursor.gold.predictions  — all model predictions with horizon
        precursor.gold.agreement    — XGBoost/TFT agreement signal
    """
    parts = [xgb_historical]
    if xgb_inference  is not None: parts.append(xgb_inference)
    if tft_historical is not None: parts.append(tft_historical)
    if tft_inference  is not None: parts.append(tft_inference)

    combined = pd.concat(parts, ignore_index=True)

    sdf = spark.createDataFrame(combined)
    sdf.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("precursor.gold.predictions")

    logger.info("precursor.gold.predictions written: %d rows", len(combined))
    for m in combined["model"].unique():
        for d in combined["dataset"].unique():
            n = len(combined[
                (combined["model"] == m) & (combined["dataset"] == d)
            ])
            logger.info("  %s / %s: %d rows", m, d, n)

    # Write agreement table
    if agreement_df is not None and len(agreement_df) > 0:
        spark.createDataFrame(agreement_df) \
            .write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("precursor.gold.agreement")
        logger.info(
            "precursor.gold.agreement written: %d rows", len(agreement_df)
        )

In [0]:
# CELL 9 — Backtest

def compute_backtest(
    set_a_xgb:      pd.DataFrame,
    set_a_tft:      pd.DataFrame,
    xgb_historical: pd.DataFrame,
    tft_historical: Optional[pd.DataFrame],
) -> pd.DataFrame:
    """Compute backtest for both models using their respective targets.

    XGBoost backtested against target_21d outcomes.
    TFT backtested against target_1d outcomes.
    """
    results = []

    model_configs = [
        ("xgboost", xgb_historical, set_a_xgb, XGB_TARGET),
        ("tft",     tft_historical, set_a_tft, TFT_TARGET),
    ]

    for model_name, preds_df, hist_df, target_col in model_configs:
        if preds_df is None:
            continue

        merged = preds_df.merge(
            hist_df[["ticker", "date", target_col, "return_1d"]],
            on=["ticker", "date"],
            how="inner",
        )
        merged = merged.dropna(subset=[target_col, "return_1d"])
        if len(merged) == 0:
            continue

        merged["take_trade"] = merged["confidence"] > 0.55
        merged["trade_return"] = np.where(
            merged["take_trade"] & (merged["prediction"] == 1),
            merged["return_1d"] - 0.0006,
            0.0,
        )
        merged["correct"] = (
            merged["prediction"] == merged[target_col]
        ).astype(int)

        daily = (
            merged.groupby("date")
            .agg(
                daily_return   =("trade_return", "mean"),
                daily_accuracy =("correct",      "mean"),
                trades_taken   =("take_trade",   "sum"),
            )
            .reset_index()
            .sort_values("date")
        )

        daily["cumulative_return"] = (
            (1 + daily["daily_return"]).cumprod() - 1
        )

        spy = hist_df[hist_df["ticker"] == "SPY"][
            ["date", "return_1d"]
        ].rename(columns={"return_1d": "spy_return"})
        daily = daily.merge(spy, on="date", how="left")
        daily["benchmark_return"] = (
            (1 + daily["spy_return"].fillna(0)).cumprod() - 1
        )

        daily["model"]        = model_name
        daily["horizon"]      = "21d" if model_name == "xgboost" else "1d"
        daily["predicted_at"] = datetime.utcnow()
        results.append(daily)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def write_backtest(backtest_df: pd.DataFrame) -> None:
    """Write backtest results to precursor.gold.backtest."""
    if len(backtest_df) == 0:
        logger.warning("Empty backtest — skipping write")
        return

    sdf = spark.createDataFrame(backtest_df)
    sdf.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("precursor.gold.backtest")
    logger.info("precursor.gold.backtest written: %d rows", len(backtest_df))

In [0]:
# CELL 10 — Findings

def compute_interesting_findings(
    set_a_xgb:      pd.DataFrame,
    xgb_historical: pd.DataFrame,
) -> dict:
    """Compute 6 analytical findings using XGBoost predictions."""
    merged = xgb_historical.merge(
        set_a_xgb[[
            "ticker", "date", "sector", XGB_TARGET,
            "macro_regime", "insider_activity_spike",
            "vix_level", "yield_curve_level"
        ]],
        on=["ticker", "date"],
        how="inner",
    ).dropna(subset=[XGB_TARGET])

    merged["correct"] = (
        merged["prediction"] == merged[XGB_TARGET]
    ).astype(int)

    findings = {}

    # Finding 1 — Macro regime
    regime_acc = merged.groupby("macro_regime")["correct"].mean().to_dict()
    findings["macro_regime_accuracy"] = {
        "accuracies": {
            "risk_off": regime_acc.get(0, None),
            "neutral":  regime_acc.get(1, None),
            "risk_on":  regime_acc.get(2, None),
        },
        "finding": (
            f"Model is "
            f"{abs((regime_acc.get(0,0.5)-regime_acc.get(2,0.5))*100):.1f}% "
            f"more accurate in risk-off vs risk-on regimes"
        ),
    }

    # Finding 2 — Insider activity
    spike_acc    = merged[merged["insider_activity_spike"]==1]["correct"].mean()
    no_spike_acc = merged[merged["insider_activity_spike"]==0]["correct"].mean()
    findings["insider_activity_edge"] = {
        "accuracy_with_spike":    float(spike_acc),
        "accuracy_without_spike": float(no_spike_acc),
        "finding": (
            f"When insider activity spikes model accuracy is "
            f"{spike_acc*100:.1f}% vs {no_spike_acc*100:.1f}% baseline"
        ),
    }

    # Finding 3 — Per-stock predictability
    stock_acc = (
        merged.groupby("ticker")["correct"].mean()
        .sort_values(ascending=False)
    )
    findings["stock_predictability"] = {
        "top_10":    stock_acc.head(10).to_dict(),
        "bottom_10": stock_acc.tail(10).to_dict(),
        "finding":   f"Most predictable: {', '.join(list(stock_acc.head(5).index))}",
    }

    # Finding 4 — VIX regime
    low_vix  = merged[merged["vix_level"] < 15]["correct"].mean()
    high_vix = merged[merged["vix_level"] > 25]["correct"].mean()
    findings["vix_regime_accuracy"] = {
        "low_vix_accuracy":  float(low_vix),
        "high_vix_accuracy": float(high_vix),
        "finding": (
            f"Model performs "
            f"{abs((low_vix-high_vix)*100):.1f}% "
            f"{'better' if low_vix > high_vix else 'worse'} "
            f"when VIX is low (<15) vs high (>25)"
        ),
    }

    # Finding 5 — Yield curve
    inv_acc    = merged[merged["yield_curve_level"] < 0]["correct"].mean()
    normal_acc = merged[merged["yield_curve_level"] >= 0]["correct"].mean()
    findings["yield_curve_signal"] = {
        "inverted_accuracy": float(inv_acc),
        "normal_accuracy":   float(normal_acc),
        "finding": (
            f"During inverted yield curve model accuracy is "
            f"{inv_acc*100:.1f}% vs {normal_acc*100:.1f}% in normal conditions"
        ),
    }

    # Finding 6 — Sector predictability
    sector_acc = (
        merged.groupby("sector")["correct"].mean()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"correct": "accuracy"})
        .to_dict(orient="records")
    )
    best  = sector_acc[0]
    worst = sector_acc[-1]
    findings["sector_predictability"] = {
        "ranking": sector_acc,
        "finding": (
            f"Most predictable: {best['sector']} "
            f"({best['accuracy']*100:.1f}%), "
            f"least: {worst['sector']} "
            f"({worst['accuracy']*100:.1f}%)"
        ),
    }

    findings_pdf = pd.DataFrame([{
        "computed_at": datetime.utcnow().isoformat(),
        "findings":    json.dumps(findings),
    }])
    spark.createDataFrame(findings_pdf) \
        .write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("precursor.gold.findings")
    logger.info("precursor.gold.findings written.")

    return findings

In [0]:
# CELL 11 — Main execution

_start = datetime.now()
logger.info("=== precursor.07_predict_v3 START at %s ===", _start.isoformat())

_set_a_xgb      = None
_set_b_xgb      = None
_set_a_tft      = None
_set_b_tft      = None
_xgb_model      = None
_xgb_inverted   = False
_tft_model      = None
_xgb_historical = None
_xgb_inference  = None
_tft_historical = None
_tft_inference  = None
_agreement_df   = None
_backtest_df    = None
_findings       = None


def _step(name: str):
    logger.info("--- Step: %s ---", name)


try:
    _step("load_features")
    _set_a_xgb, _set_b_xgb, _set_a_tft, _set_b_tft = load_features()
except Exception as exc:
    logger.error("load_features failed: %s", exc, exc_info=True)

try:
    _step("load_xgboost")
    _xgb_model, _xgb_inverted = load_xgboost()
except Exception as exc:
    logger.error("load_xgboost failed: %s", exc, exc_info=True)

try:
    _step("load_tft")
    _tft_model = load_tft()
except Exception as exc:
    logger.warning("load_tft failed: %s", exc)

try:
    _step("predict_xgboost historical")
    if _xgb_model is not None and _set_a_xgb is not None:
        _xgb_historical = predict_xgboost(
            _xgb_model, _xgb_inverted, _set_a_xgb, "historical"
        )
except Exception as exc:
    logger.error("predict_xgboost historical failed: %s", exc, exc_info=True)

try:
    _step("predict_xgboost inference")
    if _xgb_model is not None and _set_b_xgb is not None and len(_set_b_xgb) > 0:
        _xgb_inference = predict_xgboost(
            _xgb_model, _xgb_inverted, _set_b_xgb, "inference"
        )
except Exception as exc:
    logger.error("predict_xgboost inference failed: %s", exc, exc_info=True)

try:
    _step("predict_tft historical")
    if _tft_model is not None and _set_a_tft is not None:
        _tft_historical = predict_tft(_tft_model, _set_a_tft, "historical")
except Exception as exc:
    logger.warning("predict_tft historical failed: %s", exc)

try:
    _step("predict_tft inference")
    if _tft_model is not None and _set_b_tft is not None and len(_set_b_tft) > 0:
        _tft_inference = predict_tft(_tft_model, _set_b_tft, "inference")
except Exception as exc:
    logger.warning("predict_tft inference failed: %s", exc)

try:
    _step("compute_agreement")
    _agreement_df = compute_agreement(_xgb_inference, _tft_inference)
except Exception as exc:
    logger.warning("compute_agreement failed: %s", exc)

try:
    _step("write_predictions")
    if _xgb_historical is not None:
        write_predictions(
            _xgb_historical, _xgb_inference,
            _tft_historical, _tft_inference,
            _agreement_df,
        )
except Exception as exc:
    logger.error("write_predictions failed: %s", exc, exc_info=True)

try:
    _step("compute_backtest")
    if _xgb_historical is not None and _set_a_xgb is not None:
        _backtest_df = compute_backtest(
            _set_a_xgb,
            _set_a_tft if _set_a_tft is not None else _set_a_xgb,
            _xgb_historical,
            _tft_historical,
        )
except Exception as exc:
    logger.error("compute_backtest failed: %s", exc, exc_info=True)

try:
    _step("write_backtest")
    if _backtest_df is not None:
        write_backtest(_backtest_df)
except Exception as exc:
    logger.error("write_backtest failed: %s", exc, exc_info=True)

try:
    _step("compute_interesting_findings")
    if _set_a_xgb is not None and _xgb_historical is not None:
        _findings = compute_interesting_findings(_set_a_xgb, _xgb_historical)
except Exception as exc:
    logger.error("compute_interesting_findings failed: %s", exc, exc_info=True)

_elapsed = (datetime.now() - _start).total_seconds() / 60
logger.info("=== precursor.07_predict_v3 END — %.2f min ===", _elapsed)

print("\n" + "=" * 60)
print("  PRECURSOR — PREDICTION PIPELINE SUMMARY v3")
print("=" * 60)
print(f"  XGBoost target         : {XGB_TARGET} (monthly)")
print(f"  TFT target             : {TFT_TARGET} (daily)")
print(f"  XGBoost inverted       : {_xgb_inverted}")
print(f"  XGBoost historical     : {len(_xgb_historical) if _xgb_historical is not None else 0:,} rows")
print(f"  XGBoost inference      : {len(_xgb_inference)  if _xgb_inference  is not None else 0:,} rows")
print(f"  TFT historical         : {len(_tft_historical) if _tft_historical is not None else 0:,} rows")
print(f"  TFT inference          : {len(_tft_inference)  if _tft_inference  is not None else 0:,} rows")
print(f"  Agreement rows         : {len(_agreement_df) if _agreement_df is not None else 0:,}")
print(f"  Backtest rows          : {len(_backtest_df) if _backtest_df is not None else 0:,}")
print(f"  Findings computed      : {len(_findings) if _findings else 0}")
print(f"  Elapsed time           : {_elapsed:.2f} min")
print("=" * 60)

INFO:precursor.predict:=== precursor.07_predict_v3 START at 2026-05-02T23:46:30.481649 ===
INFO:precursor.predict:--- Step: load_features ---
INFO:precursor.predict:XGBoost set_a: 279644 rows  set_b: 12744 rows
INFO:precursor.predict:TFT     set_a: 290496 rows  set_b: 609 rows
INFO:precursor.predict:XGBoost inference dates: 2020-04-01 → 2026-04-30
INFO:precursor.predict:TFT inference dates: 2020-04-02 → 2026-04-30
INFO:precursor.predict:--- Step: load_xgboost ---
INFO:precursor.predict:XGBoost bundle loaded: target=target_21d auc=0.5333 inverted=False numpy=1.23.5
INFO:precursor.predict:--- Step: load_tft ---
/local_disk0/.ephemeral_nfs/envs/pythonEnv-30f073e9-3517-4ee0-9213-eff1e33fdcc9/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-30f073e9-351


  PRECURSOR — PREDICTION PIPELINE SUMMARY v3
  XGBoost target         : target_21d (monthly)
  TFT target             : target_1d (daily)
  XGBoost inverted       : False
  XGBoost historical     : 279,644 rows
  XGBoost inference      : 12,744 rows
  TFT historical         : 1,060 rows
  TFT inference          : 1,186 rows
  Agreement rows         : 399
  Backtest rows          : 1,142
  Findings computed      : 6
  Elapsed time           : 0.68 min


In [0]:
# CELL 12 — Validation

logger.info("=== Running prediction validation ===")

_checks: list[tuple[str, bool, str]] = []


def _check(name: str, passed: bool, detail: str = "") -> None:
    status = "PASS" if passed else "FAIL"
    logger.info("[%s] %s%s", status, name, f" — {detail}" if detail else "")
    _checks.append((name, passed, detail))


try:
    # predictions table
    _pred_count = spark.table("precursor.gold.predictions").count()
    _check("precursor.gold.predictions exists", _pred_count > 0,
           f"{_pred_count:,} rows")

    # models present
    _models = [
        r["model"] for r in
        spark.table("precursor.gold.predictions")
        .select("model").distinct()
        .toPandas().to_dict(orient="records")
    ]
    _check("XGBoost predictions present", "xgboost" in _models, str(_models))
    _check("TFT predictions present",     "tft"      in _models, str(_models))

    # horizons present
    _horizons = [
        r["horizon"] for r in
        spark.table("precursor.gold.predictions")
        .select("horizon").distinct()
        .toPandas().to_dict(orient="records")
    ]
    _check("21d horizon present", "21d" in _horizons, str(_horizons))
    _check("1d horizon present",  "1d"  in _horizons, str(_horizons))

    # inference rows exist
    _inf_count = (
        spark.table("precursor.gold.predictions")
        .filter("dataset = 'inference'")
        .count()
    )
    _check("Inference predictions exist", _inf_count > 0,
           f"{_inf_count:,} rows")

    # value checks
    _bad_preds = (
        spark.table("precursor.gold.predictions")
        .filter("prediction NOT IN (0, 1)").count()
    )
    _check("Prediction values only 0 or 1", _bad_preds == 0,
           f"{_bad_preds} invalid")

    _bad_prob = (
        spark.table("precursor.gold.predictions")
        .filter("probability < 0 OR probability > 1").count()
    )
    _check("Probability values in [0, 1]", _bad_prob == 0,
           f"{_bad_prob} invalid")

    # XGBoost prob variance
    _stats = (
        spark.table("precursor.gold.predictions")
        .filter("model = 'xgboost' AND dataset = 'inference'")
        .agg(
            F.stddev("probability").alias("std"),
            F.min("probability").alias("mn"),
            F.max("probability").alias("mx"),
        )
        .collect()[0]
    )
    _check("XGBoost probs vary across tickers",
           float(_stats["std"] or 0) > 0.01,
           f"std={_stats['std']:.4f} min={_stats['mn']:.4f} max={_stats['mx']:.4f}")

    # backtest
    _bt_count = spark.table("precursor.gold.backtest").count()
    _check("precursor.gold.backtest exists", _bt_count > 0,
           f"{_bt_count:,} rows")

    # findings
    _f_count = spark.table("precursor.gold.findings").count()
    _check("precursor.gold.findings exists", _f_count > 0, f"{_f_count} rows")

    # all 6 findings
    if _findings:
        _check("All 6 findings computed", len(_findings) == 6,
               f"{len(_findings)} findings")
    else:
        _check("All 6 findings computed", False, "findings is None")

    # agreement table (optional)
    try:
        _ag_count = spark.table("precursor.gold.agreement").count()
        _check("precursor.gold.agreement exists", _ag_count > 0,
               f"{_ag_count:,} rows")
    except Exception:
        _check("precursor.gold.agreement exists", False,
               "table not created (TFT may be unavailable)")

except Exception as exc:
    logger.error("Validation failed: %s", exc, exc_info=True)
    _checks.append(("Validation executed without error", False, str(exc)))

_passed_n = sum(1 for _, p, _ in _checks if p)
_failed_n = len(_checks) - _passed_n

print("=" * 60)
print("  PREDICTION VALIDATION RESULTS v3")
print("=" * 60)
for name, passed, detail in _checks:
    status = "PASS" if passed else "FAIL"
    line   = f"  [{status}] {name}"
    if detail:
        line += f"\n         {detail}"
    print(line)
print("-" * 60)
print(f"  {_passed_n} passed  /  {_failed_n} failed")
if _failed_n > 0:
    print("  WARNING: validation failures detected — review logs above.")
print("=" * 60)

INFO:precursor.predict:=== Running prediction validation ===
INFO:precursor.predict:[PASS] precursor.gold.predictions exists — 294,634 rows
INFO:precursor.predict:[PASS] XGBoost predictions present — ['xgboost', 'tft']
INFO:precursor.predict:[PASS] TFT predictions present — ['xgboost', 'tft']
INFO:precursor.predict:[PASS] 21d horizon present — ['1d', '21d']
INFO:precursor.predict:[PASS] 1d horizon present — ['1d', '21d']
INFO:precursor.predict:[PASS] Inference predictions exist — 13,930 rows
INFO:precursor.predict:[PASS] Prediction values only 0 or 1 — 0 invalid
INFO:precursor.predict:[PASS] Probability values in [0, 1] — 0 invalid
INFO:precursor.predict:[PASS] XGBoost probs vary across tickers — std=0.0182 min=0.3955 max=0.6129
INFO:precursor.predict:[PASS] precursor.gold.backtest exists — 1,142 rows
INFO:precursor.predict:[PASS] precursor.gold.findings exists — 1 rows
INFO:precursor.predict:[PASS] All 6 findings computed — 6 findings
INFO:precursor.predict:[PASS] precursor.gold.agree

  PREDICTION VALIDATION RESULTS v3
  [PASS] precursor.gold.predictions exists
         294,634 rows
  [PASS] XGBoost predictions present
         ['xgboost', 'tft']
  [PASS] TFT predictions present
         ['xgboost', 'tft']
  [PASS] 21d horizon present
         ['1d', '21d']
  [PASS] 1d horizon present
         ['1d', '21d']
  [PASS] Inference predictions exist
         13,930 rows
  [PASS] Prediction values only 0 or 1
         0 invalid
  [PASS] Probability values in [0, 1]
         0 invalid
  [PASS] XGBoost probs vary across tickers
         std=0.0182 min=0.3955 max=0.6129
  [PASS] precursor.gold.backtest exists
         1,142 rows
  [PASS] precursor.gold.findings exists
         1 rows
  [PASS] All 6 findings computed
         6 findings
  [PASS] precursor.gold.agreement exists
         399 rows
------------------------------------------------------------
  13 passed  /  0 failed


In [0]:
print(spark.conf.get("spark.databricks.workspaceUrl"))

dbc-b7ee8514-f214.cloud.databricks.com
